# 04 — Machine Learning: Fare Prediction

**Tickets:** ML-01, ML-02, ML-03, ML-04, ML-05, ML-06, ML-07  
**Business Question (BQ-3):** Can we predict the total fare of a trip before it starts?  
**Purpose:** Build a feature table, train baseline and improved models, log everything to MLflow, register the best model.

---

## Setup

In [0]:
%pip install mlflow scikit-learn --quiet

In [0]:
%restart_python

## ML-01 — Algorithm Research & Selection

### Problem Framing

**Task type:** Supervised regression — predict `total_amount` (continuous target) from features known at trip start.

**Features available before trip starts (from Silver table):**
| Feature | Type | Notes |
|---------|------|-------|
| `PULocationID` | Categorical (265 zones) | Pickup taxi zone — encode as one-hot or target-encode |
| `DOLocationID` | Categorical (265 zones) | Dropoff taxi zone (if known at dispatch) |
| `hour_of_day` | Ordinal (0-23) | Extracted from `tpep_pickup_datetime` |
| `day_of_week` | Ordinal (0-6) | Mon=0, Sun=6 |
| `is_weekend` | Binary | Saturday/Sunday flag |
| `trip_distance` | Continuous | Miles — strong predictor, but verify no data leakage |
| `passenger_count` | Discrete (1-9) | Weak predictor, but include for completeness |
| `RatecodeID` | Categorical (1-6) | Standard, JFK, Newark, etc. — very informative |

**Target:** `total_amount` (includes fare, surcharges, tips, tolls)

> **Leakage warning:** `fare_amount`, `tip_amount`, `tolls_amount`, `mta_tax`, `improvement_surcharge` are all components of `total_amount` — they must NOT be used as features.

---

### Candidate Algorithms

| # | Algorithm | Library | Pros | Cons | Expected role |
|---|-----------|---------|------|------|---------------|
| 1 | **Linear Regression** | `sklearn.linear_model.LinearRegression` | Fast to train, fully interpretable, good baseline to measure uplift against | Assumes linear feature-target relationship; struggles with interactions and non-linearity | **Baseline (ML-03)** |
| 2 | **Gradient Boosted Trees (GBT)** | `sklearn.ensemble.HistGradientBoostingRegressor` | Handles non-linearity & interactions natively; usually top performer on tabular data; feature importance built in | Slower to train; more hyperparams to tune; risk of overfitting with small data | **Improved model (ML-04)** |
| 3 | **Random Forest** | `sklearn.ensemble.RandomForestRegressor` | Robust out-of-box; less prone to overfitting than single-tree GBT; parallelisable | Typically slightly worse than tuned GBT; larger model size | **Alternative improved model or stretch (S-06)** |

#### Why these three?

- The project plan explicitly names LinearRegression, GBT, and RF as candidates.
- They cover a clear **baseline  to  improved** progression: linear  to  ensemble.
- All available via `scikit-learn` (already in `requirements.txt`), no extra dependencies needed.
- All produce feature importances (coefficients / impurity-based), useful for ML-07 interpretation.
- `HistGradientBoostingRegressor` is preferred over `GradientBoostingRegressor` for larger datasets (>10k rows) — it uses histogram-based binning for dramatically faster training with near-identical accuracy.

#### Stretch candidates (S-06)

| Algorithm | When to consider |
|-----------|-----------------|
| **Ridge / Lasso** | If Linear Regression overfits or we want regularisation |
| **XGBoost** | If `HistGradientBoosting` is not enough and we want more tuning control (requires extra install) |
| **LightGBM** | Databricks-native; fastest GBT variant; ideal if dataset is very large |

---

### Evaluation Metrics

| Metric | Why |
|--------|-----|
| **RMSE** | Primary metric — penalises large errors (important for fare estimates) |
| **MAE** | Interpretable "average dollar error" |
| **R-squared** | How much variance the model explains vs mean baseline |

All three are logged to MLflow per the project plan (ML-05).

---

### Recommended Approach

1. **ML-02:** Build feature table from Silver — select the columns above, encode categoricals, train/test split (80/20, random or stratified by `hour_of_day`).
2. **ML-03:** Train `LinearRegression` as baseline. Log to MLflow.
3. **ML-04:** Train `HistGradientBoostingRegressor` with light hyperparameter search (`max_iter`, `max_depth`, `learning_rate`). Log to MLflow.
4. **ML-05:** Compare both runs in MLflow — pick the lower-RMSE model.
5. **ML-06:** Register winner in MLflow Model Registry.
6. **ML-07:** Extract feature importances; write interpretation narrative.

If time allows, add Random Forest and/or XGBoost as stretch comparisons (S-06).

In [0]:
import importlib

import numpy as np
import mlflow
import mlflow.sklearn
import pandas as pd
import src.constants
from mlflow.models import infer_signature  # ML-06
from pyspark.sql import functions as F
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

importlib.reload(src.constants)

from src.constants import (  # noqa: E402
    SILVER_TABLE,
    ML_FEATURE_COLUMNS,
    ML_REGISTERED_MODEL_NAME,  # ML-06
    ML_SAMPLE_FRACTION,
    ML_TARGET_COLUMN,
    ML_TEST_SIZE,
    ML_RANDOM_STATE,
)

print("Setup complete")

## ML-02 — Build feature table

In [0]:
# ── Read Silver table ────────────────────────────────────────────────────────────
silver_df = spark.read.table(SILVER_TABLE)
print(f"Silver rows: {silver_df.count():,}")

# ── Select features + target ───────────────────────────────────────────────────
feature_df = silver_df.select(*ML_FEATURE_COLUMNS, ML_TARGET_COLUMN)

# Drop rows with nulls (rate_code_id NULL from code 99 mapping)
feature_df = feature_df.dropna()

# Drop rows with empty zone strings (GPS coords NULLed by clean_gps_coordinates
# produce empty strings via concat_ws)
feature_df = feature_df.filter(
    (F.col("pickup_zone") != "") & (F.col("dropoff_zone") != "")
)
print(f"After dropping nulls & empty zones: {feature_df.count():,}")

# ── Parse zone strings into numeric lat/lon bins ─────────────────────────────
# Zone format: "lat_bin,lon_bin" (e.g. "40.75,-73.99")
for prefix in ["pickup", "dropoff"]:
    zone_col = f"{prefix}_zone"
    feature_df = feature_df.withColumn(
        f"{prefix}_lat_bin", F.split(F.col(zone_col), ",")[0].cast("double")
    ).withColumn(f"{prefix}_lon_bin", F.split(F.col(zone_col), ",")[1].cast("double"))
feature_df = feature_df.drop("pickup_zone", "dropoff_zone")

# Cast is_weekend boolean → int for sklearn
feature_df = feature_df.withColumn("is_weekend", F.col("is_weekend").cast("int"))

# ── Sample for sklearn training ───────────────────────────────────────────────
# ~94M rows too large for in-memory sklearn; 1% ≈ 900K rows
feature_df = feature_df.sample(fraction=ML_SAMPLE_FRACTION, seed=ML_RANDOM_STATE)
print(f"After sampling ({ML_SAMPLE_FRACTION * 100:.0f}%): {feature_df.count():,}")

# ── Convert to pandas ─────────────────────────────────────────────────────────
pdf = feature_df.toPandas()

# ── One-hot encode rate_code_id (categorical: 1=Std, 2=JFK, 3=Newark, etc.) ──
pdf = pd.get_dummies(pdf, columns=["rate_code_id"], prefix="rc", dtype=int)

print(f"\nFeature table shape: {pdf.shape}")
feature_cols = [c for c in pdf.columns if c != ML_TARGET_COLUMN]
print(f"Features ({len(feature_cols)}): {feature_cols}")
display(pdf.head(5))

# ── Train/test split (80/20) ───────────────────────────────────────────────────
X = pdf.drop(columns=[ML_TARGET_COLUMN])
y = pdf[ML_TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=ML_TEST_SIZE, random_state=ML_RANDOM_STATE
)

print(f"\nTrain: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(f"Target (y_train) mean: ${y_train.mean():.2f}, std: ${y_train.std():.2f}")

## ML-03 — Baseline model (Linear Regression)

In [0]:
# ── Train Linear Regression baseline ───────────────────────────────────────────
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# ── Predict & evaluate ────────────────────────────────────────────────────────
y_pred_lr = lr_model.predict(X_test)

lr_metrics = {
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred_lr))),
    "mae": float(mean_absolute_error(y_test, y_pred_lr)),
    "r2": float(r2_score(y_test, y_pred_lr)),
}

# ── Log to MLflow ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_name="ML-03_LinearRegression"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_rows", X_train.shape[0])
    mlflow.log_param("test_rows", X_test.shape[0])
    mlflow.log_metrics(lr_metrics)
    mlflow.sklearn.log_model(lr_model, artifact_path="model")
    lr_run_id = mlflow.active_run().info.run_id

print("ML-03 — Linear Regression Baseline")
print("=" * 45)
print(f"  RMSE:  ${lr_metrics['rmse']:.2f}")
print(f"  MAE:   ${lr_metrics['mae']:.2f}")
print(f"  R²:    {lr_metrics['r2']:.4f}")
print(f"\nMLflow run ID: {lr_run_id}")

## ML-04 — Improved model (Gradient Boosted Trees)

In [0]:
# ── Train HistGradientBoostingRegressor ────────────────────────────────────────
# Hyperparameters chosen based on ML-01 research: HistGBT is preferred over
# vanilla GBT for large datasets (histogram-based binning, much faster).
gbt_params = {
    "max_iter": 300,
    "max_depth": 8,
    "learning_rate": 0.05,
    "min_samples_leaf": 50,
    "random_state": ML_RANDOM_STATE,
}

gbt_model = HistGradientBoostingRegressor(**gbt_params)
gbt_model.fit(X_train, y_train)

# ── Predict & evaluate ────────────────────────────────────────────────────────
y_pred_gbt = gbt_model.predict(X_test)

gbt_metrics = {
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred_gbt))),
    "mae": float(mean_absolute_error(y_test, y_pred_gbt)),
    "r2": float(r2_score(y_test, y_pred_gbt)),
}

# ── Log to MLflow ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_name="ML-04_HistGradientBoosting"):
    mlflow.log_params(gbt_params)
    mlflow.log_param("model_type", "HistGradientBoostingRegressor")
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_rows", X_train.shape[0])
    mlflow.log_param("test_rows", X_test.shape[0])
    mlflow.log_metrics(gbt_metrics)
    mlflow.sklearn.log_model(gbt_model, artifact_path="model")
    gbt_run_id = mlflow.active_run().info.run_id

print("ML-04 — HistGradientBoosting (Improved)")
print("=" * 45)
print(f"  RMSE:  ${gbt_metrics['rmse']:.2f}")
print(f"  MAE:   ${gbt_metrics['mae']:.2f}")
print(f"  R²:    {gbt_metrics['r2']:.4f}")
print(f"\nMLflow run ID: {gbt_run_id}")

## ML-05 — Compare models in MLflow

In [0]:
# ── ML-05: Compare both models ─────────────────────────────────────────────────
comparison = pd.DataFrame(
    {
        "Model": ["Linear Regression (ML-03)", "HistGradientBoosting (ML-04)"],
        "RMSE ($)": [lr_metrics["rmse"], gbt_metrics["rmse"]],
        "MAE ($)": [lr_metrics["mae"], gbt_metrics["mae"]],
        "R²": [lr_metrics["r2"], gbt_metrics["r2"]],
        "MLflow Run ID": [lr_run_id, gbt_run_id],
    }
)

print("ML-05 — Model Comparison")
print("=" * 70)
display(comparison)

# ── Pick the winner by lowest RMSE ────────────────────────────────────────────
if gbt_metrics["rmse"] < lr_metrics["rmse"]:
    best_model_name = "HistGradientBoostingRegressor"
    best_model = gbt_model
    best_run_id = gbt_run_id
    best_metrics = gbt_metrics
    rmse_improvement = lr_metrics["rmse"] - gbt_metrics["rmse"]
else:
    best_model_name = "LinearRegression"
    best_model = lr_model
    best_run_id = lr_run_id
    best_metrics = lr_metrics
    rmse_improvement = gbt_metrics["rmse"] - lr_metrics["rmse"]

print(f"\n✅ Best model: {best_model_name}")
print(
    f"   RMSE: ${best_metrics['rmse']:.2f}  |  MAE: ${best_metrics['mae']:.2f}  |  R²: {best_metrics['r2']:.4f}"
)
print(f"   RMSE improvement over runner-up: ${rmse_improvement:.2f}")
print(f"   MLflow run ID: {best_run_id}")

## ML-06 — Register best model

In [0]:
# ── ML-06: Register best model in Unity Catalog ────────────────────────────────

# Point the registry at Unity Catalog (not the legacy workspace registry)
mlflow.set_registry_uri("databricks-uc")

# Infer model signature from training data so the serving UI and endpoint
# can validate inputs and auto-generate API documentation.
signature = infer_signature(X_train, best_model.predict(X_train))

# Provide a small input example for the Model Registry UI
input_example = X_train[:5].copy()

# Re-log the winning model WITH signature + input_example and register
# in a single step.  ML-03/ML-04 logged without these, so a fresh run
# is needed for UC compatibility.
with mlflow.start_run(run_name=f"ML-06_{best_model_name}_registered"):
    mlflow.log_param("model_type", best_model_name)
    mlflow.log_param("source_run_id", best_run_id)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_rows", X_train.shape[0])
    mlflow.log_param("test_rows", X_test.shape[0])
    mlflow.log_metrics(best_metrics)

    model_info = mlflow.sklearn.log_model(
        best_model,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        registered_model_name=ML_REGISTERED_MODEL_NAME,
    )

    ml06_run_id = mlflow.active_run().info.run_id

print("ML-06 — Model Registration")
print("=" * 55)
print(f"  ✅ Registered to: {ML_REGISTERED_MODEL_NAME}")
print(f"  Model:         {best_model_name}")
print(f"  Source run:    {best_run_id}")
print(f"  Registry run:  {ml06_run_id}")
print(f"  Model URI:     {model_info.model_uri}")
print(f"  RMSE: ${best_metrics['rmse']:.2f}  |  MAE: ${best_metrics['mae']:.2f}  |  R²: {best_metrics['r2']:.4f}")

## ML-07 — Model interpretation

<!-- Summarise what the model learned: top features, directional relationships, limitations -->